In [1]:
"""
Create an income raster for Nigeria using OnStove income_estimation.
This script uses only method 1 (AWE estimation) in this notebook.
Method 2 (Income CSV interpolation) is intentionally commented out.
"""

import os
import shutil
from onstove import OnStove

# User parameters (paths relative to this notebook folder)
model_pickle_path = os.path.normpath("../../example/NGA/model_inputs.pkl")
scenario_csv_path = os.path.normpath("../../example/NGA/soc_specs.csv")
output_directory = "dataset_big"

# Method 2 not used in this notebook (kept as reference)
# use_income_csv = True
# income_csv_path = "dataset_big/nigeria_income_percentiles.csv"

# Required for AWE mode
gini_value = 0.339      # Example: 0.35
gdp_pc_value = 2139    # Example: 2200.0
pareto_weight = 0.32

# Output raster column and name (AWE writes absolute_wealth)
output_variable = "absolute_wealth"
output_raster_name = "income_nigeria"

# Basic input checks
if not os.path.exists(model_pickle_path):
    raise FileNotFoundError(f"Model pickle not found: {model_pickle_path}")
if not os.path.exists(scenario_csv_path):
    raise FileNotFoundError(f"Scenario csv not found: {scenario_csv_path}")
os.makedirs(output_directory, exist_ok=True)

# 1) Load prepared model and scenario
model = OnStove.read_model(model_pickle_path)
model.read_scenario_data(scenario_csv_path, delimiter=",")
model.output_directory = output_directory

# 2) Configure inputs for selected income mode
# AWE only branch
if gini_value is None or gdp_pc_value is None:
    raise ValueError("Set gini_value and gdp_pc_value for AWE mode")
model.specs["gini"] = float(gini_value)
model.specs["gdp_pc"] = float(gdp_pc_value)
model.income_estimation(awe=True, income_data=None, pareto_weight=pareto_weight)

# 3) Quick checks
if "relative_wealth" not in model.gdf.columns:
    raise KeyError("Model gdf is missing 'relative_wealth', required by income_estimation")
if output_variable not in model.gdf.columns:
    raise KeyError(f"Expected output column '{output_variable}' not found in model.gdf")

# 4) Export raster (suppresses OnStove's internal print statements)
import contextlib
import io
with contextlib.redirect_stdout(io.StringIO()):
    model.to_raster(variable=output_variable)

# Move file to final destination and clean up
src_name = f"{output_variable}_mean.tif"
src_path = os.path.join(output_directory, "Rasters", src_name)
dst_path = os.path.join(output_directory, f"{output_raster_name}.tif")
if os.path.exists(src_path):
    if os.path.exists(dst_path):
        os.remove(dst_path)
    os.replace(src_path, dst_path)
    print(f"Raster moved to: {dst_path}")

# Delete the temporary Rasters directory
rasters_dir = os.path.join(output_directory, "Rasters")
if os.path.exists(rasters_dir):
    shutil.rmtree(rasters_dir)
    print(f"Temporary directory deleted: {rasters_dir}")

print("Income mode: AWE")
print(f"Final raster: {dst_path if os.path.exists(dst_path) else 'NOT FOUND'}")


Raster moved to: dataset_big\income_nigeria.tif
Temporary directory deleted: dataset_big\Rasters
Income mode: AWE
Final raster: dataset_big\income_nigeria.tif


In [2]:
"""
Reproject income raster to EPSG:3857 (Web Mercator), normalize, and create vehicle_possibility.
Vehicle possibility = normalized income with exponent applied.
"""

import rioxarray
import numpy as np
import xarray as xr
from rasterio.warp import Resampling

def main():
    
    # User parameters
    input_path = "dataset_big/income_nigeria.tif"
    output_vehicle_path = "dataset_big/vehicle_possibility.tif"
    target_crs = "EPSG:3857"
    target_resolution = 1000  # meters in EPSG:3857 (1 km)
    output_nodata = -9999.0

    # Shape parameter for vehicle possibility (applied on normalized income).
    # 1.0 keeps linear relation, >1 emphasizes high-income pixels, <1 flattens differences.
    income_exponent = 1.0
    
    # Read the income raster in its original CRS (already on Nigeria from OnStove)
    print(f"Reading income raster from {input_path}...")
    income_da = rioxarray.open_rasterio(input_path)
    original_crs = income_da.rio.crs
    original_nodata = income_da.rio.nodata
    print(f"Original CRS: {original_crs}")
    print(f"Original resolution: {income_da.rio.resolution()}")
    print(f"Original nodata value: {original_nodata}")
    print(f"Array contains NaN values: {np.isnan(income_da.values).any()}")
    
    # Reproject to target CRS
    print(f"Reprojecting to {target_crs} with resolution {target_resolution} m...")
    income_3857 = income_da.rio.reproject(
        target_crs,
        resolution=target_resolution,
        resampling=Resampling.nearest
    )
    income_3857 = income_3857.fillna(output_nodata)
    income_3857.rio.write_nodata(output_nodata, inplace=True)
    
    # Normalize income to [0, 1] and apply exponent to create vehicle_possibility.
    print("Normalizing income to [0, 1] and applying exponent...")
    data = income_3857.values.astype(np.float32, copy=True)
    valid = (np.isfinite(data)) & (data != output_nodata)
    if original_nodata is not None:
        valid = valid & (data != original_nodata)
    
    if valid.any():
        vmin = data[valid].min()
        vmax = data[valid].max()
        
        if vmax > vmin:
            data[valid] = (data[valid] - vmin) / (vmax - vmin)
        else:
            data[valid] = 0.0
        
        # Apply exponent on already normalized data
        data[valid] = np.clip(data[valid], 0.0, 1.0)
        data[valid] = np.power(data[valid], income_exponent, dtype=np.float32)
        
        data[~valid] = output_nodata
        
        print(f"Data normalized and exponent applied: min={float(data[valid].min()):.4f}, max={float(data[valid].max()):.4f}")
        print(f"Nodata value={output_nodata}, nodata pixels={int((~valid).sum())}")
    else:
        raise ValueError("No valid pixels found for normalization.")
    
    # Build vehicle_possibility raster (single output)
    vehicle_possibility = xr.DataArray(
        data,
        coords=income_3857.coords,
        dims=income_3857.dims,
        attrs=income_3857.attrs,
        name="vehicle_possibility",
    )
    vehicle_possibility.rio.write_crs(target_crs, inplace=True)
    vehicle_possibility.rio.write_nodata(output_nodata, inplace=True)
    vehicle_possibility.attrs["source"] = "income_only"
    vehicle_possibility.attrs["income_exponent"] = float(income_exponent)

    print(f"Writing vehicle possibility raster to {output_vehicle_path}...")
    vehicle_possibility.rio.to_raster(output_vehicle_path, compress="DEFLATE", nodata=output_nodata)

    out = vehicle_possibility.values
    out_valid = np.isfinite(out) & (out != output_nodata)
    print(f"Done. Vehicle possibility raster saved as {output_vehicle_path} in {target_crs}")
    print(f"Approach: income normalized with exponent={income_exponent}")
    print(f"Output range: min={float(out[out_valid].min()):.4f}, max={float(out[out_valid].max()):.4f}")
    
if __name__ == "__main__":
    main()



Reading income raster from dataset_big/income_nigeria.tif...
Original CRS: EPSG:3395
Original resolution: (1000.0, -1000.0)
Original nodata value: 0.0
Array contains NaN values: False
Reprojecting to EPSG:3857 with resolution 1000 m...
Normalizing income to [0, 1] and applying exponent...
Data normalized and exponent applied: min=0.0000, max=1.0000
Nodata value=-9999.0, nodata pixels=802386
Writing vehicle possibility raster to dataset_big/vehicle_possibility.tif...
Done. Vehicle possibility raster saved as dataset_big/vehicle_possibility.tif in EPSG:3857
Approach: income normalized with exponent=1.0
Output range: min=0.0000, max=1.0000


In [3]:
"""
Vehicle-access allocation rationale

This cell distributes a fixed national target of car access across pixels
in Nigeria, combining:
1) vehicle_possibility (derived from income only)
2) population per pixel.

Method idea:
- the pixel with the highest vehicle_possibility is constrained to 100% access,
- all other pixels receive a lower access share, scaled smoothly,
- a calibration parameter (alpha) is found so that the sum of allocated people
  with car access is exactly equal to the national target,
- very small modeled access shares are floored to zero using a minimum share
  threshold during calibration and final allocation.

National target definition:
- reference vehicles = 11,600,000 [OICA, 2020]
- average users sharing one vehicle
- target people with access = shared_users_per_vehicle * reference vehicles

Final outputs are three separate rasters (names kept for compatibility):
- vehicles_allocation_share.tif: share (%) of population with car access in each pixel
- walk_allocation_share.tif: share (%) of population without car access in each pixel
- vehicles_allocation_n_effettivo.tif: number of people with car access in each pixel


VALUE OF AVERAGE USERS PER VEHICLE AND CONSIDERATIONS

For the Nigeria case study, I recommend using 5.06 persons per household as the national reference value, 
with a clear urban-rural split of 4.50 persons in urban areas and 5.42 persons in rural areas, based on the 
Nigeria Living Standards Survey 2020 by the National Bureau of Statistics. Since direct data on car-sharing 
per vehicle is rarely available, this household size is a reasonable proxy for the number of people 
potentially sharing one private car, especially when the model needs a consistent population-based assumption.
https://www.nigerianstat.gov.ng/elibrary/read/1123

For extrapolation across Nigeria, these values can be applied as the baseline assumption for 
household-based vehicle access, while noting that they represent an approximation rather than 
a measured car-occupancy statistic. If a broader regional benchmark is needed for Sub-Saharan 
Africa, a useful indicative range is about 5–6 persons in urban areas and 6–7 persons in rural 
areas, which is consistent with demographic patterns reported in the literature on African 
household structure. based on demographic analyses drawing from the United Nations Population 
Division database on household size and composition and DHS‑type surveys covering the region.
https://www.un.org/development/desa/pd/household-size-and-composition
"""

import numpy as np
import xarray as xr
import rioxarray
from rasterio.warp import Resampling

# Inputs
vehicle_possibility_path = "dataset_big/vehicle_possibility.tif"
population_path = "dataset_big/Population.tif"
output_share_car_access_path = "dataset_big/vehicles_allocation_share.tif"
output_share_walk_access_path = "dataset_big/walk_allocation_share.tif"
output_cars_path = "dataset_big/vehicles_allocation_n_effettivo.tif"

urban_raster_path = "dataset_big/Urban.tif"        # values: 0‑100 (urbanisation degree)
URBAN_THRESHOLD = 20                                # cells >= 20 considered urban

reference_total_vehicles = 11_600_000
f_urban = 4.50      # users_urban
f_rural = 5.42      # users_rural
target_vehicles = reference_total_vehicles   #target


output_nodata = -9999.0
min_vehicle_share = 1e-5

def total_vehicles_for_alpha(alpha, s_norm, pop, F_arr, min_share):
    """Return total allocated vehicles for a given alpha after share flooring."""
    rates = np.power(s_norm, alpha, dtype=np.float64)
    rates = np.where(rates < min_share, 0.0, rates)
    vehicles = pop * rates / F_arr
    return float(np.sum(vehicles, dtype=np.float64))

# 1) Read rasters and align population to the vehicle-possibility grid
score_da = rioxarray.open_rasterio(vehicle_possibility_path)
pop_da = rioxarray.open_rasterio(population_path)
pop_da = pop_da.rio.reproject_match(score_da, resampling=Resampling.nearest)

# Read urban raster and align to the same grid
urban_da = rioxarray.open_rasterio(urban_raster_path)
urban_da = urban_da.rio.reproject_match(score_da, resampling=Resampling.nearest)


score_nodata = score_da.rio.nodata
pop_nodata = pop_da.rio.nodata


score = score_da.values.astype(np.float64, copy=False)
pop = pop_da.values.astype(np.float64, copy=False)

urban = urban_da.values.astype(np.float64, copy=False)
urban_nodata = urban_da.rio.nodata



# 2) Build valid mask: finite, positive population, and non-nodata values
valid = (np.isfinite(score) & np.isfinite(pop) & np.isfinite(urban) & (pop > 0))
if score_nodata is not None:
    valid &= score != score_nodata
if pop_nodata is not None:
    valid &= pop != pop_nodata
if urban_nodata is not None:
    valid &= urban != urban_nodata

if not valid.any():
    raise ValueError("No valid pixels found after masking.")

# 3) Convert vehicle possibility to [0, 1] among valid pixels
s = score[valid]
s_min = float(np.min(s))
s_max = float(np.max(s))

if s_max <= s_min:
    raise ValueError("Vehicle possibility has no variation on valid pixels.")

s_norm = (s - s_min) / (s_max - s_min)
s_norm = np.clip(s_norm, 0.0, 1.0)

# Force one max pixel to exactly 1.0 => 100% access in that pixel
max_idx = int(np.argmax(s))
s_norm[max_idx] = 1.0

pop_valid = pop[valid]


urban_valid = urban[valid]
urban_flag = urban_valid >= URBAN_THRESHOLD
F_valid = np.where(urban_flag, f_urban, f_rural).astype(np.float64)

# 4) Calibrate alpha with bisection so “sum(vehicles) = target_vehicles”.

max_vehicles_at_alpha0 = total_vehicles_for_alpha(0.0, s_norm, pop_valid, F_valid, min_vehicle_share)
if target_vehicles > max_vehicles_at_alpha0:
    raise ValueError(
        f"Requested target_vehicles={target_vehicles:,} exceeds maximum theoretical capacity={max_vehicles_at_alpha0:,.0f}."
    )

left, right = 0.0, 40.0
for _ in range(80):
    mid = 0.5 * (left + right)
    veh_mid = total_vehicles_for_alpha(mid, s_norm, pop_valid, F_valid, min_vehicle_share)
    if veh_mid > target_vehicles:
        left = mid
    else:
        right = mid
alpha = 0.5 * (left + right)

# 5) Compute final per-pixel outputs
raw_rates_valid = np.power(s_norm, alpha, dtype=np.float64)
rates_valid = np.where(raw_rates_valid < min_vehicle_share, 0.0, raw_rates_valid)
zeroed_small_positive = int(np.sum((raw_rates_valid > 0.0) & (raw_rates_valid < min_vehicle_share)))

vehicles_valid = pop_valid * rates_valid / F_valid
total_allocated_vehicles = float(np.sum(vehicles_valid))
access_people_valid = pop_valid * rates_valid

share_car_access_percent = np.full(score.shape, output_nodata, dtype=np.float32)
share_walk_access_percent = np.full(score.shape, output_nodata, dtype=np.float32)
n_effettivo = np.full(score.shape, output_nodata, dtype=np.float32)

share_car_access_percent[valid] = (rates_valid * 100.0).astype(np.float32)
share_walk_access_percent[valid] = (100.0 - share_car_access_percent[valid]).astype(np.float32)
n_effettivo[valid] = access_people_valid.astype(np.float32)

# 6) Save to separate GeoTIFF files (same grid, same CRS)
share_car_access_da = xr.DataArray(
    share_car_access_percent,
    coords=score_da.coords,
    dims=score_da.dims,
    name="share_car_access",
)
share_car_access_da.rio.write_crs(score_da.rio.crs, inplace=True)
share_car_access_da.rio.write_nodata(output_nodata, inplace=True)
share_car_access_da.attrs["long_name"] = "vehicles_allocation_share_percent"
share_car_access_da.attrs["reference_total_vehicles"] = int(reference_total_vehicles)
share_car_access_da.attrs["alpha"] = float(alpha)
share_car_access_da.attrs["min_vehicle_share"] = float(min_vehicle_share)
share_car_access_da.attrs["f_urban"] = float(f_urban)
share_car_access_da.attrs["f_rural"] = float(f_rural)
share_car_access_da.attrs["target_vehicles"] = int(target_vehicles)
share_car_access_da.attrs["allocated_vehicles"] = float(total_allocated_vehicles)   
share_car_access_da.rio.to_raster(output_share_car_access_path, compress="DEFLATE", nodata=output_nodata)



share_walk_access_da = xr.DataArray(
    share_walk_access_percent,
    coords=score_da.coords,
    dims=score_da.dims,
    name="share_walk_access",
)
share_walk_access_da.rio.write_crs(score_da.rio.crs, inplace=True)
share_walk_access_da.rio.write_nodata(output_nodata, inplace=True)
share_walk_access_da.attrs["long_name"] = "walk_allocation_share_percent"
share_walk_access_da.attrs["derived_from"] = "100 - vehicles_allocation_share"
share_walk_access_da.attrs["reference_total_vehicles"] = int(reference_total_vehicles)
share_walk_access_da.attrs["f_urban"] = float(f_urban)
share_walk_access_da.attrs["f_rural"] = float(f_rural)
share_walk_access_da.attrs["target_vehicles"] = int(target_vehicles)
share_walk_access_da.attrs["allocated_vehicles"] = float(total_allocated_vehicles)
share_walk_access_da.attrs["alpha"] = float(alpha)
share_walk_access_da.attrs["min_vehicle_share"] = float(min_vehicle_share)
share_walk_access_da.rio.to_raster(output_share_walk_access_path, compress="DEFLATE", nodata=output_nodata)

cars_da = xr.DataArray(
    n_effettivo,
    coords=score_da.coords,
    dims=score_da.dims,
    name="n_effettivo",
)
cars_da.rio.write_crs(score_da.rio.crs, inplace=True)
cars_da.rio.write_nodata(output_nodata, inplace=True)
cars_da.attrs["long_name"] = "vehicle_access_allocated_people"
cars_da.attrs["reference_total_vehicles"] = int(reference_total_vehicles)
cars_da.attrs["f_urban"] = float(f_urban)
cars_da.attrs["f_rural"] = float(f_rural)
cars_da.attrs["target_vehicles"] = int(target_vehicles)
cars_da.attrs["allocated_vehicles"] = float(total_allocated_vehicles)
cars_da.attrs["alpha"] = float(alpha)
cars_da.attrs["min_vehicle_share"] = float(min_vehicle_share)
cars_da.rio.to_raster(output_cars_path, compress="DEFLATE", nodata=output_nodata)

# 7) Minimal summary of allocation result
total_allocated_people = float(np.sum(n_effettivo[n_effettivo != output_nodata], dtype=np.float64))



# Gap veicoli (dovrebbe essere molto piccolo per via della calibrazione)
vehicle_gap = total_allocated_vehicles - target_vehicles
vehicle_gap_pct = (vehicle_gap / target_vehicles) * 100.0

effective_f = total_allocated_people / total_allocated_vehicles if total_allocated_vehicles > 0 else 0.0
max_share_car_access = float(np.max(share_car_access_percent[share_car_access_percent != output_nodata]))
min_share_walk_access = float(np.min(share_walk_access_percent[share_walk_access_percent != output_nodata]))


print(f"Estimated alpha: {alpha:.6f}")
print(f"Reference total vehicles: {reference_total_vehicles:,.0f}")
print(f"F urban: {f_urban}, F rural: {f_rural}")
print(f"Target vehicles: {target_vehicles:,.0f}")
print(f"Allocated vehicles: {total_allocated_vehicles:,.0f}")
print(f"Vehicle gap (allocated - target): {vehicle_gap:+,.3f} ({vehicle_gap_pct:+.6f}%)")
print(f"Total people with car access: {total_allocated_people:,.0f}")
print(f"Effective national average users/vehicle: {effective_f:.2f}")
print(f"Min vehicle share threshold applied: {min_vehicle_share:.1e}")
print(f"Pixels floored to zero by threshold: {zeroed_small_positive:,}")
print(f"Max car access share (%): {max_share_car_access:.2f}")
print(f"Min walk access share (%): {min_share_walk_access:.2f}")
print(f"Car share raster written: {output_share_car_access_path}")
print(f"Walk share raster written: {output_share_walk_access_path}")
print(f"Access-people raster written: {output_cars_path}")

Estimated alpha: 1.401241
Reference total vehicles: 11,600,000
F urban: 4.5, F rural: 5.42
Target vehicles: 11,600,000
Allocated vehicles: 11,600,000
Vehicle gap (allocated - target): +0.000 (+0.000000%)
Total people with car access: 54,446,415
Effective national average users/vehicle: 4.69
Min vehicle share threshold applied: 1.0e-05
Pixels floored to zero by threshold: 0
Max car access share (%): 100.00
Min walk access share (%): 0.00
Car share raster written: dataset_big/vehicles_allocation_share.tif
Walk share raster written: dataset_big/walk_allocation_share.tif
Access-people raster written: dataset_big/vehicles_allocation_n_effettivo.tif


In [4]:
# Quick stats on car-ownership share distribution per pixel

import numpy as np
import rioxarray

share_path = "dataset_big/vehicles_allocation_share.tif"
share_da = rioxarray.open_rasterio(share_path)

nodata = share_da.rio.nodata
data = share_da.values.astype(np.float64, copy=False)

valid = np.isfinite(data)
if nodata is not None:
    valid &= data != nodata

vals = data[valid]

if vals.size == 0:
    raise ValueError("No valid values found in share raster.")

print("=== SHARE DISTRIBUTION (% auto per pixel) ===")
print(f"Valid cells: {vals.size:,}")
print(f"Min: {vals.min():.2f}%")
print(f"P10: {np.percentile(vals, 10):.2f}%")
print(f"P25: {np.percentile(vals, 25):.2f}%")
print(f"Median: {np.percentile(vals, 50):.2f}%")
print(f"Mean: {vals.mean():.2f}%")
print(f"P75: {np.percentile(vals, 75):.2f}%")
print(f"P90: {np.percentile(vals, 90):.2f}%")
print(f"P95: {np.percentile(vals, 95):.2f}%")
print(f"Max: {vals.max():.2f}%")

# Bins in percentage points
bins = np.array([0, 1, 5, 10, 20, 40, 60, 80, 95, 100.0001], dtype=np.float64)
labels = [
    "0-1%", "1-5%", "5-10%", "10-20%", "20-40%",
    "40-60%", "60-80%", "80-95%", "95-100%"
]

counts, _ = np.histogram(vals, bins=bins)
shares = counts / vals.size * 100.0

print("\n=== CELLS BY SHARE CLASS ===")
for lbl, c, s in zip(labels, counts, shares):
    print(f"{lbl:>8}: {c:>8,} cells ({s:>6.2f}%)")

# Extra: how many cells are near full adoption
above_90 = np.sum(vals >= 90)
above_50 = np.sum(vals >= 50)
print("\n=== QUICK THRESHOLDS ===")
print(f"Cells >= 50%: {above_50:,} ({above_50/vals.size*100:.2f}%)")
print(f"Cells >= 90%: {above_90:,} ({above_90/vals.size*100:.2f}%)")

=== SHARE DISTRIBUTION (% auto per pixel) ===
Valid cells: 560,635
Min: 0.00%
P10: 1.94%
P25: 2.90%
Median: 4.78%
Mean: 7.92%
P75: 8.98%
P90: 16.52%
P95: 24.11%
Max: 100.00%

=== CELLS BY SHARE CLASS ===
    0-1%:   17,241 cells (  3.08%)
    1-5%:  274,405 cells ( 48.95%)
   5-10%:  147,603 cells ( 26.33%)
  10-20%:   81,430 cells ( 14.52%)
  20-40%:   30,255 cells (  5.40%)
  40-60%:    5,655 cells (  1.01%)
  60-80%:    1,826 cells (  0.33%)
  80-95%:    1,105 cells (  0.20%)
 95-100%:    1,115 cells (  0.20%)

=== QUICK THRESHOLDS ===
Cells >= 50%: 5,980 (1.07%)
Cells >= 90%: 1,527 (0.27%)
